# Импорт библиотек
Импортируются необходимые библиотеки для работы с chunking, базой данных chromadb, pandas и sentence-transformers.

In [1]:
from chonkie import RecursiveChunker, Visualizer
from chromadb import PersistentClient
from chromadb.api.models import Collection
import pandas as pd
from sentence_transformers import SentenceTransformer
from tqdm import tqdm

In [ ]:
chuck = RecursiveChunker()


# Обёртка для коллекции ChromaDB
Определяется класс-обёртка и функция для удобной работы с коллекциями ChromaDB, включая автоматическое удаление коллекции после выхода из контекста.

In [3]:
class ChromaCollectionWrapper:
    def __init__(self, name, **kwargs):
        self.name = name
        self.kwargs = kwargs
        self.client = PersistentClient("./data/chroma")
        self.collection = self.client.get_or_create_collection(name, **kwargs)

    def __enter__(self) -> Collection.Collection:
        return self.collection

    def __exit__(self, exc_type, exc_val, exc_tb):
        print("for debug")
        self.client.delete_collection(self.name)

    def __getattr__(self, item):
        return getattr(self.collection, item)

def chroma_collection(name, **kwargs):
    '''
    Универсальный способ получить коллекцию chromadb.
    Можно использовать как с with, так и без него.
    Если используется с with, коллекция удаляется после выхода.
    '''
    return ChromaCollectionWrapper(name, **kwargs)

# Функция оценки качества поиска
Функция evaluate вычисляет метрику MRR для поиска по embedding'ам.

In [21]:
def evaluate(model, questions, collection, total_docs):
    # model - embedding model
    # questions - evaluation questions
    # total_docs - for n_results return

    columns = ["question", "position", "score"]
    row_data = []

    for _, row in questions.iterrows():
        question = row["question"]
        y_true = row["page_id"]
        embedding = model.encode(question, prompt_name='search_document')

        results = collection.query(embedding, n_results=total_docs)

        for i, position in enumerate(results["metadatas"][0], start=1):
            
            if position["page_id"] == y_true:
                data_row = [question, i, 1/i]
                row_data.append(data_row)
        
                break
      
        else:
            data_row = [question, 0, 0]
            row_data.append(data_row)
    
    df = pd.DataFrame(row_data, columns=columns)
    df = pd.concat([df, pd.DataFrame(data={"question": "MRR", "position": df.score.sum()/df.shape[0], "score": "-------"}, index=[df.shape[0]])], axis=0)

    print(df)
    return df


# Функции предобработки текста
Здесь определены функции для обработки заголовков, извлечения и преобразования таблиц, а также удаления изображений из текста.

In [5]:
import re

def preprocess_headings(text):
    # Заменяем h1., h2., ... h6. на соответствующее количество #
    def repl(match):
        level = int(match.group(1))
        return '\n' + ('#' * level) + ' '
    return re.sub(r'\bh([1-6])\.\s*', repl, text)


def extract_tables_and_text(text):
    """
    Извлекает все таблицы из текста, преобразует их в markdown и возвращает:
    - текст без таблиц
    - список таблиц в markdown-формате
    """
    # Находит все таблицы по шаблону: строки начинаются и заканчиваются на '|'
    table_pattern = r'(?:\n)?((?:\|.*\|\n?)+)'
    tables = re.findall(table_pattern, text)
    markdown_tables = []

    for table in tables:
        lines = [line.strip() for line in table.strip().split('\n') if line.strip()]
        if not lines:
            continue
        header = lines[0]
        columns = [col.strip() for col in header.strip('|').split('|')]
        separator = '|' + '|'.join(['---'] * len(columns)) + '|'
        markdown_table = '\n'.join([header, separator] + lines[1:])
        markdown_tables.append(markdown_table)

    # Удаляем таблицы из текста
    text_without_tables = re.sub(table_pattern, '', text).strip()

    return text_without_tables, markdown_tables

def replace_tables_with_markdown(text):
    """
    Находит все таблицы в тексте и заменяет их на markdown-таблицы с разделителем после заголовка.
    Возвращает изменённый текст.
    """
    def table_to_markdown(table):
        lines = [line.strip() for line in table.strip().split('\n') if line.strip()]
        if not lines:
            return ''
        header = lines[0]
        columns = [col.strip() for col in header.strip('|').split('|')]
        separator = '|' + '|'.join(['---'] * len(columns)) + '|'
        return '\n'.join([header, separator] + lines[1:])

    table_pattern = r'((?:\|.*\|\n?)+)'
    def replacer(match):
        return table_to_markdown(match.group(1))

    return re.sub(table_pattern, replacer, text)

def remove_image_tags(text):
    """
    Удаляет все конструкции вида {{...расширение}} (png, jpg, jpeg, gif, webp, bmp, svg)
    """
    return re.sub(r'\{\{[^{}]*?\.(png|jpg|jpeg|gif|webp|bmp|svg)\}\}|\{\{undefined\}\}', '', text, flags=re.IGNORECASE)


Добавил функцию для замены ссылок на изображения на [IMG_{здесь мог быть ваш индекс}]. Также функцию восстановления, дальше по коду пока не используется, потому что незачем.

Также функция убирает \n перед ссылками, чтобы чанкеры справлялись лучше

In [6]:
import re
from typing import Tuple, Dict

def preprocess_images(text: str) -> Tuple[str, Dict[str, str]]:
    """
    Заменяет ссылки на изображения на плейсхолдеры [IMG_N],
    при этом удаляя все \n перед вставкой (до ближайшего не-\n символа).
    """
    pattern = r'!\{[^}]*\}[^!\s]+!?|![^!\s]+!|\{\{[^}]+\}\}'
    replacements = {}
    clean_text = text
    offset = 0  # смещение из-за изменения длины текста

    for match in re.finditer(pattern, text):
        start, end = match.start() - offset, match.end() - offset
        placeholder = f"[IMG_{len(replacements) + 1}]"
        replacements[placeholder] = match.group(0)

        # --- удаляем \n и пробелы перед вставкой ---
        back = start
        while back > 0 and clean_text[back - 1] in ['\n', '\r', ' ']:
            back -= 1
        # но если всё до начала — не трогаем
        if back < start:
            clean_text = clean_text[:back] + placeholder + clean_text[end:]
            offset += (end - back) - len(placeholder)
        else:
            clean_text = clean_text[:start] + placeholder + clean_text[end:]
            offset += (end - start) - len(placeholder)

    return clean_text, replacements


def restore_images(text: str, replacements: Dict[str, str]) -> str:
    """Восстанавливает оригинальные вставки по плейсхолдерам."""
    for placeholder, original in replacements.items():
        text = text.replace(placeholder, original)
    return text


# Загрузка данных и подготовка embedding-модели
Загружаются вопросы и тексты, а также инициализируется модель для получения эмбеддингов.

Ещё немного потыкал другие модельки, пока сложно что-то конкртеное сказать, надо проверять

In [6]:
# model = "ai-forever/ru-en-RoSBERTa"
# model = "deepvk/USER2-base"
# model = "sergeyzh/rubert-mini-frida"
# embeddings = SentenceTransformer(model)

q = pd.read_csv("./dataset/questions.csv")
docs = pd.read_csv("./dataset/texts.csv")
raw_texts = []
for index, row in docs.iterrows():
    with open(f"./dataset/texts/{row['page_id']}.txt", "r") as f:
        raw_texts.append(f.read())

docs["text"] = raw_texts

# Chunking markdown-текста с RecursiveChunker
Разделение markdown-текста на чанки с помощью RecursiveChunker.

Был 0.96 на 28 документах, сейчас 0.79

In [28]:
from chonkie import RecursiveChunker

ids = []
documents = []
embeds = []
metadatas = []

chunker = RecursiveChunker(
    chunk_size = 128
).from_recipe("markdown", lang="en")

for n, (index, row) in enumerate(tqdm(docs.iterrows(), total=len(docs)), start=1):
    text = row['text']

    for n_chunk, chunk in enumerate(chunker.chunk(text), start=1):
        ids.append(f"id_{n}_{n_chunk}")
        documents.append(chunk.text)
        embeds.append(embeddings.encode(chunk.text))
        metadatas.append({"page_id": row["page_id"], "chunk_id": n_chunk})

with chroma_collection("test_markdown") as test:
    test.add(ids=ids, documents=documents, embeddings=embeds, metadatas=metadatas)
    evaluate(embeddings, q, test, len(docs))

100%|██████████| 172/172 [21:07<00:00,  7.37s/it]


                                             question   position     score
0    Посещение завершено, как закрыть случай лечения?   2.000000       0.5
1                   Как оформить  направление на МСЭ?   1.000000       1.0
2   Как создать направление на диагностическое исс...   1.000000       1.0
3                                   Как создать МКСБ?   1.000000       1.0
4   Как оформить направление на плановую госпитали...   3.000000  0.333333
5            Как перейти в «План иммунопрофилактики»?   1.000000       1.0
6                    Кака создавать реестры в ВебМИС?   2.000000       0.5
7   Какую роль нужно добавить пользователю, чтобы ...   1.000000       1.0
8                             Как создать новый МКАБ?   2.000000       0.5
9   Как проверить факт прикрепления пациента в сис...  12.000000  0.083333
10  Какие поля являются обязательными при заполнен...   1.000000       1.0
11    Как распечатать согласия пациента в стационаре?   1.000000       1.0
12  Как выдать заключение

# Chunking markdown-текста с SemanticChunker
Разделение markdown-текста на чанки с помощью SemanticChunker.

In [ ]:
from chonkie import SemanticChunker

ids = []
documents = []
embeds = []
metadatas = []

chunker = SemanticChunker(
    embedding_model = model,
    chunk_size = 256
).from_recipe("markdown", lang="en")

for n, (index, row) in enumerate(tqdm(docs.iterrows(), total=len(docs)), start=1):
    text = row['text']


    for n_chunk, chunk in enumerate(chunker.chunk(text), start=1):
        ids.append(f"id_{n}_{n_chunk}")
        documents.append(chunk.text)
        embeds.append(embeddings.encode(chunk.text))
        metadatas.append({"page_id": row["page_id"], "chunk_id": n_chunk})

with chroma_collection("test_markdown") as test:
    test.add(ids=ids, documents=documents, embeddings=embeds, metadatas=metadatas)
    evaluate(embeddings, q, test, len(docs))

Some weights of RobertaModel were not initialized from the model checkpoint at ai-forever/ru-en-RoSBERTa and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
  5%|▌         | 9/172 [03:40<1:01:11, 22.53s/it]/home/egor/repo/Learning-nlp-models/.venv/lib64/python3.12/site-packages/chonkie/embeddings/model2vec.py:64: RuntimeWarning: invalid value encountered in divide
  return np.divide(
100%|██████████| 172/172 [30:26<00:00, 10.62s/it] 


                                             question   position     score
0    Посещение завершено, как закрыть случай лечения?   8.000000     0.125
1                   Как оформить  направление на МСЭ?   1.000000       1.0
2   Как создать направление на диагностическое исс...   1.000000       1.0
3                                   Как создать МКСБ?   1.000000       1.0
4   Как оформить направление на плановую госпитали...  33.000000  0.030303
5            Как перейти в «План иммунопрофилактики»?   1.000000       1.0
6                    Кака создавать реестры в ВебМИС?   3.000000  0.333333
7   Какую роль нужно добавить пользователю, чтобы ...   1.000000       1.0
8                             Как создать новый МКАБ?   2.000000       0.5
9   Как проверить факт прикрепления пациента в сис...  32.000000   0.03125
10  Какие поля являются обязательными при заполнен...   1.000000       1.0
11    Как распечатать согласия пациента в стационаре?   2.000000       0.5
12  Как выдать заключение

# Chunking markdown-текста с предобработкой с RecursiveChunker
Замена изображеий из markdown-текста и разбиение на чанки с помощью RecursiveChunker.

In [22]:
from chonkie import RecursiveChunker

ids = []
documents = []
embeds = []
metadatas = []

chunker = RecursiveChunker(
    chunk_size = 128
).from_recipe("markdown", lang="en")

for n, (index, row) in enumerate(tqdm(docs.iterrows(), total=len(docs)), start=1):
    text = remove_image_tags(row['text'])
    clear_text, placeholders = preprocess_images(text)

    for n_chunk, chunk in enumerate(chunker.chunk(clear_text), start=1):
        ids.append(f"id_{n}_{n_chunk}")
        documents.append(chunk.text)
        embeds.append(embeddings.encode(chunk.text, prompt_name='search_document'))
        metadatas.append({"page_id": row["page_id"], "chunk_id": n_chunk})

with chroma_collection("test_markdown") as test:
    test.add(ids=ids, documents=documents, embeddings=embeds, metadatas=metadatas)
    evaluate(embeddings, q, test, len(docs))

100%|██████████| 172/172 [16:35<00:00,  5.79s/it]


                                             question   position     score
0    Посещение завершено, как закрыть случай лечения?  20.000000      0.05
1                   Как оформить  направление на МСЭ?   1.000000       1.0
2   Как создать направление на диагностическое исс...   2.000000       0.5
3                                   Как создать МКСБ?   1.000000       1.0
4   Как оформить направление на плановую госпитали...  11.000000  0.090909
5            Как перейти в «План иммунопрофилактики»?   1.000000       1.0
6                    Кака создавать реестры в ВебМИС?   2.000000       0.5
7   Какую роль нужно добавить пользователю, чтобы ...   1.000000       1.0
8                             Как создать новый МКАБ?   3.000000  0.333333
9   Как проверить факт прикрепления пациента в сис...  64.000000  0.015625
10  Какие поля являются обязательными при заполнен...   3.000000  0.333333
11    Как распечатать согласия пациента в стационаре?   4.000000      0.25
12  Как выдать заключение

# Chunking markdown-текста с предобработкой с SemanticChunker
Замена изображеий из markdown-текста и разбиение на чанки с помощью SemanticChunker.

In [9]:
from chonkie import SemanticChunker, AutoEmbeddings

ids = []
documents = []
embeds = []
metadatas = []

chunker = SemanticChunker(
    embedding_model = AutoEmbeddings.get_embeddings(model),
    chunk_size = 256
).from_recipe("markdown", lang="en")

for n, (index, row) in enumerate(tqdm(docs.iterrows(), total=len(docs)), start=1):
    text = remove_image_tags(row['text'])
    clear_text, placeholders = preprocess_images(text)

    for n_chunk, chunk in enumerate(chunker.chunk(clear_text), start=1):
        ids.append(f"id_{n}_{n_chunk}")
        documents.append(chunk.text)
        embeds.append(embeddings.encode(chunk.text, prompt_name='search_document'))
        metadatas.append({"page_id": row["page_id"], "chunk_id": n_chunk})

with chroma_collection("test_markdown") as test:
    test.add(ids=ids, documents=documents, embeddings=embeds, metadatas=metadatas)
    evaluate(embeddings, q, test, len(docs))

Some weights of RobertaModel were not initialized from the model checkpoint at ai-forever/ru-en-RoSBERTa and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
  1%|          | 2/172 [01:30<2:18:26, 48.86s/it]/home/egor/repo/Learning-nlp-models/.venv/lib64/python3.12/site-packages/chonkie/embeddings/model2vec.py:64: RuntimeWarning: invalid value encountered in divide
  return np.divide(
100%|██████████| 172/172 [27:07<00:00,  9.46s/it] 


                                             question   position     score
0    Посещение завершено, как закрыть случай лечения?  18.000000  0.055556
1                   Как оформить  направление на МСЭ?   1.000000       1.0
2   Как создать направление на диагностическое исс...   2.000000       0.5
3                                   Как создать МКСБ?   3.000000  0.333333
4   Как оформить направление на плановую госпитали...   2.000000       0.5
5            Как перейти в «План иммунопрофилактики»?   1.000000       1.0
6                    Кака создавать реестры в ВебМИС?   2.000000       0.5
7   Какую роль нужно добавить пользователю, чтобы ...   1.000000       1.0
8                             Как создать новый МКАБ?   1.000000       1.0
9   Как проверить факт прикрепления пациента в сис...   9.000000  0.111111
10  Какие поля являются обязательными при заполнен...   6.000000  0.166667
11    Как распечатать согласия пациента в стационаре?   2.000000       0.5
12  Как выдать заключение

# Визуализация чанков SemanticChunker
Визуализация результата разбиения текста на чанки с помощью SemanticChunker и Visualizer.

In [58]:
from chonkie import SemanticChunker

chunker = SemanticChunker(
    embedding_model = model,
    chunk_size = 256
).from_recipe("markdown", lang="en")

Visualizer()(chunker.chunk(remove_image_tags(docs["text"][6])))

Some weights of RobertaModel were not initialized from the model checkpoint at ai-forever/ru-en-RoSBERTa and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Реестры

h1. Реестры ОМС


h2. Описание



В +реестр медицинских услуг в системе ОМС+ включаются все услуги, которые были оказаны пациентам при получении ими 
бесплатной медицинской помощи по программе обязательного медицинского страхования в лечебно-профилактическом 
учреждении. Формирование реестра ОМС осуществляется согласно Генерального тарифного соглашения Территориального 
фонда обязательного медицинского страхования.

h2. Типы реестров



| h1. Наименование реестра* | *Описание* |
| Поликлиника, Стационар | Законченные случаи оказанной медицинской помощи, кроме высокотехнологичной медицинской 
помощи, медицинской помощи по диспансеризации, профилактическим медицинским осмотрам несовершеннолетних и 
профилактическим медицинским осмотрам взрослого населения, медицинской помощи при подозрении на злокачественное 
новообразование или установленном диагнозе злокачественного новообразования |
| ДД ОГВН 1 этап | Законченные случаи на оплату медицинской помощи, оказанной застрахованному лицу в рамках первого
этапа диспансеризации определенных групп взрослого населения; |
| ДД ОГВН 2 этап | Законченные случаи на оплату медицинской помощи, оказанной застрахованному лицу в рамках второго
этапа диспансеризации определенных групп взрослого населения; |
| УД ОГВН 1 этап | Законченные случаи на оплату медицинской помощи, оказанной застрахованному лицу в рамках первого
этапа *углубленной* диспансеризации определенных групп взрослого населения, перенесших COVID-19; |
| УД ОГВН 2 этап | Законченные случаи на оплату медицинской помощи, оказанной застрахованному лицу в рамках второго
этапа *углубленной* диспансеризации определенных групп взрослого населения, перенесших COVID-19; |
| Профосмотры взрослого населения | Законченные случаи на оплату медицинской помощи, оказанной застрахованному лицу
в рамках профилактических осмотров взрослого населения |
| ДДС | Законченные случаи на оплату медицинской помощи, оказанной застрахованному лицу в рамках диспансеризации 
пребывающих в стационарных учреждениях детей-сирот и детей, находящихся в трудной жизненной ситуации |
| ДДС (опека) | Законченные случаи на оплату медицинской помощи, оказанной застрахованному лицу в рамках 
диспансеризации детей-сирот и детей, оставшихся без попечения родителей, в том числе усыновленных (удочеренных), 
принятых под опеку (попечительство), в приемную или патронатную семью |
| Профилактический медосмотр несовершеннолетних | Законченные случаи на оплату медицинской помощи, оказанной 
застрахованному лицу в рамках профилактических медицинских осмотров несовершеннолетних |
| Поликлиника (ЗНО), Стационар (ЗНО) | Законченные случаи оказанной медицинской помощи при подозрении на 
злокачественное новообразование или установленном диагнозе злокачественного новообразования |
| ВМП | Законченные случаи оказанной высокотехнологичной медицинской помощи |

h2. Виды реестров



*Ежемесячно* подаются реестры на оплату:
* Поликлиника 
* Стационар
* Поликлиника (ЗНО)
* Стационар (ЗНО)
* ДД ОГВН 1 этап
* ДД ОГВН 2 этап
* Профосмотры взрослого населения
* ДДС
* ДДС (опека)
* Профилактический медосмотр несовершеннолетних
* ВМП

*Еженедельно* подаются реестры на оплату:
* УД ОГВН 1 этап
* УД ОГВН 2 этап

*В соответствии с приказом ФФОМС №79 (Приложение Д)* файлы реестров именуются особым образом.
*Пример* наименований файлов реестров, которые *ежемесячно* отправляются от МО в ТФОМС:

Архив HM280012T28_21081.zip, в котором находятся все файлы реестра для ежемесячной подачи:
HM280012T28_21081.xml - Основной (Общий)
DPM280012T28_21081.xml - ДД ОГВН 1 этап
DVM280012T28_21081.xml - ДД ОГВН 2 этап
DOM280012T28_21081.xml - Профосмотры взрослого населения
DSM280012T28_21081.xml - ДДС
DUM280012T28_21081.xml - ДДС (опека)
DFM280012T28_21081.xml - Профилактический медосмотр несовершеннолетних
CM280012T28_21081.xml - ОНКО
TM280012T28_21081.xml - ВМП

Пример наименований файлов реестров, которые еженедельно отправляются от МО в ТФОМС:

Архив DAM280012T28_21081.zip, в котором находятся все файлы реес

# Визуализация чанков RecursiveChunker
Визуализация результата разбиения текста на чанки с помощью RecursiveChunker и Visualizer.

In [20]:
from chonkie import Visualizer

chunker = RecursiveChunker(
    chunk_size = 128
).from_recipe("markdown", lang="en")

Visualizer()(chunker.chunk(docs.loc[27, 'text']))

WEB Регистратура: Работа с МКАБ в синей версии

h1. Работа с МКАБ в синей версии

МКАБ создается в зеленой МИС ("инструкция по созданию 
МКАБ":https://sd.hostco.ru/projects/amurmis/wiki/%D0%A1%D0%BE%D0%B7%D0%B4%D0%B0%D0%BD%D0%B8%D0%B5_%D0%B8_%D1%80%D0%
B5%D0%B4%D0%B0%D0%BA%D1%82%D0%B8%D1%80%D0%BE%D0%B2%D0%B0%D0%BD%D0%B8%D0%B5_%D0%9C%D0%9A%D0%90%D0%91), чтобы перейти
в редактировании в синей МКАБ, найдите карту нужного пациента, нажмите правой кнопкой мыши и выберете «Посмотреть 
МКАБ».
{{4743cb09-1b1e-49cc-bf49-ddd8a676fa24.bmp}}

В новой вкладке откроется МКАБ выбранного пациента.

{{d3e404e4-3db2-4d0a-a7f1-bbf526adca17.bmp}}

Слева находится панель разделов МКАБ. С её помощью удобно быстро переходить по разделам. 
{{bcdcdba3-104c-4091-92ab-d71b8e635db5.bmp}}


h3. Персональные данные 

Для редактирования персональных данных нажмите на «карандаш» {{691588ee-c33f-47ca-a2a9-c0ab6be62fae.bmp}},  который
расположен рядом с именем пациента.

При редактировании персональных данных есть 4 основных подраздела: 

h1.  Основная информация, где указываются данные пациента.
* Полисы, где отображаются полисы 
* Прикрепления, где отображаются прикрепления пациента по МО
* Дополнительная информация, где указывается доп. сведения по пациенту (сведения о работе и учебе, показатели 
здоровья, представители, согласия и тд)
{{61c19f8d-c8bf-40cf-b123-5a95a925bed3.bmp}}

Поля, помеченные звездочкой {{b64cd6ad-25b6-4c9b-a7df-27ddb25c3b9d.bmp}}  обязательны для заполнения.
В основной информации заполняются данные пациента: Номер МКАБ, ФИО, дата рождения, соц. статус и тд.
{{cb6ff3d5-0989-4528-8f3c-c5afa5a9574a.bmp}}
{{68193685-769d-4a37-889f-a530323a466c.bmp}}
{{0d5543af-1671-4c17-a857-3575e6ed3bf4.bmp}}

После заполнения или редактирования данных нажмите {{1b4c771b-5856-429e-83a4-f91a641dac27.bmp}},  если изменение 
данных не требуется, то {{e77b7313-9e65-4872-ad9c-65b1f8e143f9.bmp}} .


h3. Представитель пациента

Если у пациента есть представители, то добавьте их, нажав {{46c4c73b-95ce-444f-a23a-cfdfa8728f4b.bmp}}, в разделе 
«представители».

Заполнить данные. Если у представителя имеется МКАБ, добавьте её с помощью 
{{dd706119-dbf6-436d-991e-c18f13b80702.bmp}}, найдите представителя и выберете его. После чего данные представителя
заполнятся автоматически. 
Проставить галочки, напротив того, кем является представитель.
{{7d9e7cb5-ef41-43ce-81d3-416208f1ced4.bmp}}

После заполнения информации нажмите {{1b4c771b-5856-429e-83a4-f91a641dac27.bmp}}, для добавления данных, либо 
{{e77b7313-9e65-4872-ad9c-65b1f8e143f9.bmp}}, если сохранение данных не требуется.

h3. Расположение карты

Расположение карты, показывает движение МКАБ. Чтобы указать новые данные по движению карты, нажмите 
{{46c4c73b-95ce-444f-a23a-cfdfa8728f4b.bmp}}, заполните данные и сохраните изменения.

{{995b3154-922c-47e8-ae55-47e33e4e7b9f.bmp}}
{{f5054fca-ca80-45cd-866d-f0d70bad8273.bmp}}
Отправитель указывается автоматически. Есть возможность выбора отправителя вручную.

h3. Прикрепления

Прикрепления, показывает участки прикрепления пациента. Добавьте их, нажав на 
{{46c4c73b-95ce-444f-a23a-cfdfa8728f4b.bmp}}, внесите данные и сохраните. 

{{59f0d02e-6082-4e4e-83de-a399ec11c84c.bmp}}
{{bce1ea44-0643-45c4-b3b6-71f9e0ffc1a6.bmp}}

h3. Дополнительная информация

В дополнительной информации добавьте информацию о потенциально-опасных социальных и/или рабочих факторах, если 
данные были предоставлены.

{{cb8445d7-2746-41d3-a40c-da58abda919d.bmp}}
{{95decb83-ea03-4905-8dbd-96dc82eef3b9.bmp}}

h3. Информация о занятии спортом

Следующий раздел — это информация о занятии спортом.

{{089d6ceb-0dce-43d4-85e5-7633580c38e9.bmp}}

Через {{46c4c73b-95ce-444f-a23a-cfdfa8728f4b.bmp}} внесите данные о занятиях спортом. 
{{beb674de-6f3a-4e52-a996-9dcfc924511b.bmp}}
Заполните информацию, проставьте галки, если это основной спорт и участник соревнований и нажмите 
{{1b4c771b-5856-429e-83a4-f91a641dac27.bmp}}, если нужно отменить всё, то нажмите 
{{e77b7313-9e65-4872-ad9c-65b1f8e143f9.bmp}}. 
После сох

In [6]:
from chonkie import Visualizer

chunker = RecursiveChunker(
    chunk_size = 128
).from_recipe("markdown", lang="en")

Visualizer()(chunker.chunk(preprocess_images(preprocess_headings(docs.loc[0, 'text']))[0]))
# preprocess_images(docs.loc[27, 'text'])[1]

WEB Диспансеризация


# Диспансеризация в web-версии МИС



## Настройка ролей

Перед непосредственной работой по оформлению медосмотров и диспансеризаций пациентов должна быть осуществлена 
настройка системы в части медицинских обследований.
Для настройки данного модуля, необходимо назначить ответственному лицу роли: 
 -  Региональный администратор (Диспансеризация)
 -  Медицинские обследования (Администрирование)
 -  Медицинские обследования
Для создания маршрутных листов регистратором или кабинетом Профилактики и работе врача с картой мед.обследования 
назначить роль «Медицинские обследования».


## Настройка модуля «Профилактика»

Настройка модуля «Профилактика» начинается с раздела «Диспансеризация»[IMG_1]

Сопоставьте мероприятия с ресурсами (кабинеты, врачи, оборудование), которые будут выполнять мероприятия. Для этого
переходим в раздел «Мероприятия и ресурсы»
Загрузится страница сопоставления, где вы осуществляете поиск по «Виду медицинского обследования» и нажимаете 
кнопку «Найти[IMG_2]

Например, произведем настройку для «Водительской справки А, В, М»
После нажатия на кнопку «Найти» появится список всех мероприятий[IMG_3]

Теперь производим сопоставление каждого мероприятия с ресурсом через кнопку редактировать:[IMG_4]

Вводим начальные символы ресурса и выбираем ресурс, после всего выбора обязательно нажмите кнопку 
«Сохранить»[IMG_5]

Если ресурса нет, то его не указываете, далее можно указать, что выполнено ранее в другом МО.

# Обязательное указание ресурса для мероприятий с видом "Анкетирование и Прием врача"* (Смена вида мероприятия 
осуществляется по заявке сотрудниками Хост).


## Создание расписания для записи на медосмотры и диспансеризацию

После сопоставления всех мероприятий с ресурсами, необходимо создать расписание на каждый ресурс с типами 
«Медосмотр» (для записи на медицинский осмотр) и «Диспансеризация» (для записи на диспансеризацию)[IMG_6]


## Формирование маршрутного листа и запись в расписание

Регистратор или ответственное лицо создает «Маршрутный лист» в расписании на прием[IMG_7]

Находят пациента по ФИО/СНИЛС/Полису/Номеру карты и нажимают кнопку «Выбрать»[IMG_8]

Открывается окно для формирования «Маршрутного листа», где выбирается «План» и период прохождения 
мероприятий[IMG_9]

*Важно! По периоду система ищет свободные слоты в расписании ресурсов.*

Далее нажимаете кнопку «Подобрать мероприятия»[IMG_10] После чего будет сформирован список мероприятий по 
выбранному плану.

Если мероприятие было сделано ранее, то нажмите кнопку[IMG_11] и проставьте дату прохождения.[IMG_12]

Далее вы формируете маршрутный лист по кнопке[IMG_13] В маршрутном листе все мероприятия распределяются по ресурсам
на доступное время. При желании можно запись перенести.[IMG_14]

В итоге вы печатаете маршрутный лист по кнопке «Печать»[IMG_15]

Далее нажимаете «Сохранить и закрыть».


## Формирование маршрутного листа без записи в расписание

Создать Маршрутный лист можно без привязки к расписанию. Для этого на главном экране заходим в раздел "Расписание 
приема"[IMG_16]

В боковом меню выбираем «Маршрутный лист»[IMG_17]

В открывшемся окне находим нужного нам пациента и нажимаем кнопку «Выбрать»[IMG_18]

Откроется окно создания маршрутного листа, в котором необходимо выбрать План (1), модель (2) и нажать кнопку 
Подобрать мероприятие (3)[IMG_19]

Далее откроется маршрутный лист со всеми мероприятиями, после чего нажмите кнопку «сформировать маршрутный 
лист»[IMG_20]

Сформируется маршрутный лист, напротив мероприятия будет указан тип записи «Самозапись»[IMG_21]

Нажмите кнопку «Сохранить и закрыть». 
Далее для работы с картой перейдите в блок «Карты медицинских обследований»[IMG_22]

В открывшемся списке выберите нужную карту[IMG_23]

Далее процесс заполнения карты не отличается от заполнения карты с привязкой к расписанию. 


## Перенести мероприятие на другое время или изменить ресурс

1. Открываем Маршрутный лист
2. На необходимом мероприятии нажимаем на кнопку "Перенести запись" 
3. В открывшейся форме перено

In [9]:
from chonkie import RecursiveChunker

ids = []
documents = []
embeds = []
metadatas = []

chunker = RecursiveChunker(
    chunk_size = 128
).from_recipe("markdown", lang="en")

for n, (index, row) in enumerate(tqdm(docs.iterrows(), total=len(docs)), start=1):
    text = row['text']
    clear_text, placeholders = preprocess_images(text)

    for n_chunk, chunk in enumerate(chunker.chunk(clear_text), start=1):
        ids.append(f"id_{n}_{n_chunk}")
        documents.append(chunk.text)
        embeds.append(embeddings.encode(chunk.text))
        metadatas.append({"page_id": row["page_id"], "chunk_id": n_chunk})



100%|██████████| 172/172 [17:49<00:00,  6.22s/it]


In [ ]:
with chroma_collection("result_view") as view:
    view.add(ids=ids, documents=documents, embeddings=embeds, metadatas=metadatas)
    result = view.query(embeddings.encode(q.loc[9, "question"]), n_results=50)
    print(result)

{'ids': [['id_27_1', 'id_31_2', 'id_29_1', 'id_27_2', 'id_47_1', 'id_88_2', 'id_27_4', 'id_56_1', 'id_4_2', 'id_56_3', 'id_41_12', 'id_78_2', 'id_56_7', 'id_23_1', 'id_29_2', 'id_38_1', 'id_21_3', 'id_2_1', 'id_15_1', 'id_55_1', 'id_4_3', 'id_54_4', 'id_19_2', 'id_1_3', 'id_166_3', 'id_54_1', 'id_118_7', 'id_25_2', 'id_14_1', 'id_2_11', 'id_134_2', 'id_41_2', 'id_69_2', 'id_2_10', 'id_26_1', 'id_11_2', 'id_40_2', 'id_3_1', 'id_28_8', 'id_78_1', 'id_2_7', 'id_6_1', 'id_50_1', 'id_89_2', 'id_9_1', 'id_87_6', 'id_46_2', 'id_54_3', 'id_163_2', 'id_42_1']], 'embeddings': None, 'documents': [['WEB Регистратура: Прикрепление через ЕПГУ и по личному заявлению пациента в МО\n\nh1. Прикрепление через ЕПГУ и по личному заявлению пациента в МО\n\n\nДля работы с заявлениями на прикрепление, полученных с Единого портала государственных услуг (ЕПГУ), назначьте пользователю роли: \nh1.  Оператор рассмотрения заявлений на прикрепление \n* Отклонение заявок на прикрепление\n\nРоль добавляется через блок

In [19]:
display(result['metadatas'][0][38], result['metadatas'][0][1])
display(result['distances'][0][38], result['distances'][0][1])

{'chunk_id': 8, 'page_id': 28}

{'page_id': 31, 'chunk_id': 2}

0.7670572996139526

0.611464262008667

In [28]:
# Multi-objective: минимизировать близость к док.28 и максимизировать близость к док.26
import optuna
import requests
from sklearn.metrics.pairwise import cosine_distances
from chonkie import RecursiveChunker, SemanticChunker, AutoEmbeddings

config_url = "https://huggingface.co/ai-forever/ru-en-RoSBERTa/resolve/main/config_sentence_transformers.json"
config = requests.get(config_url).json()
prompt_names = list(config.get("prompts", {}).keys())
prompt_names.append(None)

question = q.loc[9, "question"]  # 10-й вопрос (визуальный, но используем .loc[9])
doc_idx_pos = 27  # 28-й документ: минимизируем расстояние
doc_idx_far = 26  # 27-й документ: максимизируем расстояние
document_pos = docs.loc[doc_idx_pos, "text"]
document_far = docs.loc[doc_idx_far, "text"]

# chunker = RecursiveChunker(
#     chunk_size = 128
# ).from_recipe("markdown", lang="en")
# chunks_pos = [chunk.text for chunk in chunker.chunk(preprocess_images(document_pos)[0])]
# chunks_far = [chunk.text for chunk in chunker.chunk(preprocess_images(document_far)[0])]

def objective(trial):
    prompt_name_q = trial.suggest_categorical("prompt_name_q", prompt_names)
    prompt_name_d = trial.suggest_categorical("prompt_name_d", prompt_names)
    prompt_name_c = trial.suggest_categorical("prompt_name_c", prompt_names)

    chunker = SemanticChunker(
        embedding_model = AutoEmbeddings.get_embeddings(model, default_prompt_name=prompt_name_c),
        chunk_size = 256
    ).from_recipe("markdown", lang="en")
    chunks_pos = [chunk.text for chunk in chunker.chunk(preprocess_images(document_pos)[0])]
    chunks_far = [chunk.text for chunk in chunker.chunk(preprocess_images(document_far)[0])]
    q_emb = embeddings.encode(question, prompt_name=prompt_name_q)

    d_embs_pos = [embeddings.encode(ch, prompt_name=prompt_name_d) for ch in chunks_pos]
    dists_pos = [cosine_distances([q_emb], [d_emb])[0][0] for d_emb in d_embs_pos]
    min_dist_pos = min(dists_pos)  # хотим МИНИМИЗИРОВАТЬ

    d_embs_far = [embeddings.encode(ch, prompt_name=prompt_name_d) for ch in chunks_far]
    dists_far = [cosine_distances([q_emb], [d_emb])[0][0] for d_emb in d_embs_far]
    min_dist_far = min(dists_far)  # хотим МАКСИМИЗИРОВАТЬ

    return min_dist_pos, min_dist_far

study = optuna.create_study(directions=["minimize", "maximize"])
study.optimize(objective, n_trials=100, show_progress_bar=True)

# Выберем трейл с минимальным первым значением и максимальным вторым
best_trial = sorted(study.best_trials, key=lambda t: (t.values[0], -t.values[1]))[0]
print("Лучшие параметры:", best_trial.params)
print("min_dist(док.28):", best_trial.values[0], "| min_dist(док.26):", best_trial.values[1])

[I 2025-10-28 23:07:27,431] A new study created in memory with name: no-name-0fa9eb36-f44a-4387-b232-3e379081aa7a


  0%|          | 0/100 [00:00<?, ?it/s]

Some weights of RobertaModel were not initialized from the model checkpoint at ai-forever/ru-en-RoSBERTa and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Default prompt name is set to 'classification'. This prompt will be applied to all `encode()` calls, except if `encode()` is called with `prompt` or `prompt_name` parameters.


[I 2025-10-28 23:08:16,924] Trial 0 finished with values: [0.39389312267303467, 0.35339784622192383] and parameters: {'prompt_name_q': 'clustering', 'prompt_name_d': 'classification', 'prompt_name_c': 'classification'}.


Some weights of RobertaModel were not initialized from the model checkpoint at ai-forever/ru-en-RoSBERTa and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Default prompt name is set to 'search_query'. This prompt will be applied to all `encode()` calls, except if `encode()` is called with `prompt` or `prompt_name` parameters.


...

[I 2025-10-29 00:27:26,291] Trial 97 finished with values: [0.3335721492767334, 0.32739245891571045] and parameters: {'prompt_name_q': 'clustering', 'prompt_name_d': 'search_query', 'prompt_name_c': 'search_query'}.


Some weights of RobertaModel were not initialized from the model checkpoint at ai-forever/ru-en-RoSBERTa and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Default prompt name is set to 'search_document'. This prompt will be applied to all `encode()` calls, except if `encode()` is called with `prompt` or `prompt_name` parameters.


[I 2025-10-29 00:28:13,240] Trial 98 finished with values: [0.3033456802368164, 0.284728467464447] and parameters: {'prompt_name_q': 'classification', 'prompt_name_d': 'clustering', 'prompt_name_c': 'search_document'}.


Some weights of RobertaModel were not initialized from the model checkpoint at ai-forever/ru-en-RoSBERTa and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Default prompt name is set to 'search_query'. This prompt will be applied to all `encode()` calls, except if `encode()` is called with `prompt` or `prompt_name` parameters.


[I 2025-10-29 00:28:59,169] Trial 99 finished with values: [0.3093114495277405, 0.2849506735801697] and parameters: {'prompt_name_q': None, 'prompt_name_d': 'clustering', 'prompt_name_c': 'search_query'}.
Лучшие параметры: {'prompt_name_q': 'search_document', 'prompt_name_d': 'search_document', 'prompt_name_c': 'search_document'}
min_dist(док.28): 0.17335808277130127 | min_dist(док.26): 0.1972205638885498


In [29]:
import optuna.visualization

optuna.visualization.plot_param_importances(study).show()

In [4]:
import re
from typing import List

def find_abbreviations(text: str) -> List[str]:
    """
    Ищет аббревиатуры в тексте. Аббревиатуры определяются как слова, содержащие
    заглавные буквы, возможно, со строчными буквами, но с обязательным наличием
    хотя бы одной заглавной буквы.

    Args:
        text (str): Текст для поиска.

    Returns:
        List[str]: Список найденных аббревиатур.
    """
    pattern = r'\b(?:[A-ZА-ЯЁ][a-zа-яё]*[A-ZА-ЯЁ]+[a-zа-яё]*|[A-ZА-ЯЁ]{2,})\b'
    return re.findall(pattern, text)

# Пример использования
example_text = "В тексте встречаются аббревиатуры, такие как ВебМИС, NASA, МИС, и т.д."
abbreviations = find_abbreviations(example_text)
print("Найденные аббревиатуры:", abbreviations)

Найденные аббревиатуры: ['ВебМИС', 'NASA', 'МИС']


In [25]:
for i, text in enumerate(docs['text'].values, 1):
    print(find_abbreviations(text), i)

['WEB', 'МИС', 'МО', 'ФИО', 'СНИЛС', 'ТАП', 'ТАП', 'ЭМД', 'ТАП', 'ТАП', 'SCORE', 'ЗНО', 'ЗНО', 'ЭМД', 'II', 'II', 'ПМО', 'МКАБ', 'ФИО', 'МКАБ', 'ПМО', 'ПМО', 'ПМО', 'ПМО', 'ПМО'] 1
['WEB', 'ТАП', 'ТАП', 'ТАП', 'МИС', 'ТАП', 'ТАП', 'COVID', 'МЗ', 'МКБ', 'МЗ', 'МЗ', 'МЗ', 'МО', 'CITO', 'МЗ', 'МЗ', 'ФИО', 'МО', 'ФИО', 'ДУ', 'ДН', 'ДУ', 'ЛН', 'ЛС', 'МЗ', 'МНН', 'ЛС', 'ЛС', 'ЛС', 'ЛС', 'ТМ', 'МИС', 'SaaS', 'ТМ', 'МИС', 'SaaS', 'МСЭ', 'ВМП', 'СМП', 'ТАП', 'ТАП', 'ТАП', 'ТАП', 'МКБ', 'МО', 'СЭМД', 'МО', 'МО', 'МО', 'ВНИМАНИЕ', 'МО', 'МО', 'ЗНО', 'ТАП', 'ЗНО', 'МКБ', 'ЗНО', 'ЗНО', 'ЗНО', 'ЗНО', 'ЗНО', 'TNM', 'ЗНО', 'ЭМД', 'МЗ', 'ЗНО', 'МКБ', 'ЗНО', 'TNM', 'IV', 'IVA', 'IVB', 'IVC', 'МКБ', 'III', 'IIIA', 'IIIB', 'IIIC', 'IIID', 'МЗ', 'ВИМИС', 'ЗНО', 'ЗНО', 'ЗНО', 'ЗНО', 'ЗНО', 'ВК', 'ВК', 'ВК', 'МКБ', 'ВК', 'ВК', 'ВК', 'ВК', 'ВК', 'СЭМД', 'ВК', 'ТАП', 'ТАП', 'ТАП', 'МКАБ', 'МИС', 'BE', 'BA', 'BB', 'BE', 'BF', 'BA', 'BA', 'BE', 'BC', 'BC', 'BA', 'BB', 'BE', 'BD', 'BF', 'BE', 'AD', 'BF', 'BD', 'B

In [21]:
terms = pd.read_csv("dataset/terms.csv")
terms

,name,meaning,description
0,АПП,Амбулаторно-поликлиническая помощь,"Вид медицинской помощи, оказываемой больным на..."
1,ВИМИС,Вертикально-интегрированная медицинская информ...,Создается в дополнение к региональным МИС и ЕГ...
2,ИЭМК,Интегрированная электронная медицинская карта,Совокупность электронных персональных медицинс...
3,КЛАДР,Классификатор адресов Российской Федерации,Объектами классификации являются отдельные эле...
4,МКАБ,Медицинская карта амбулаторного больного,"Cущность в Программе, которая соответствует ут..."
5,МКСБ,Медицинская карта стационарного больного,"Cущность в Программе, которая соответствует ут..."
6,МСЭ,Медико-социальная экспертиза,"Исследование, проводимое специальной комиссией..."
7,МСР,Медико-социальная реабилитация,"Комплекс медицинских мер, направленных на прео..."
8,МУ,Медицинские услуги,"Значения, созданные в ТАПе на вкладке ""Мед. за..."
9,НСИ,Нормативно-справочная информация,"Ведение и поддержка различных справочников, ко..."


In [22]:
def replace_abbreviations(text: str, terms: pd.DataFrame) -> str:
    """
    Заменяет аббревиатуры в тексте на их полные формулировки, используя DataFrame terms.

    Args:
        text (str): Исходный текст.
        terms (pd.DataFrame): DataFrame с колонками "abbreviation" и "full_form".

    Returns:
        str: Текст с заменёнными аббревиатурами.
    """
    for _, row in terms.iterrows():
        abbreviation = row["name"]
        full_form = row["meaning"]
        # Используем регулярное выражение для точного совпадения аббревиатуры
        text = re.sub(rf'\b{re.escape(abbreviation)}\b', full_form, text)
    return text

# Пример использования
example_text = "NASA и МИС активно развиваются."
processed_text = replace_abbreviations(example_text, terms)
print(processed_text)

NASA и МИС активно развиваются.


In [ ]:
docs.loc[35, 'text']

'WEB Родовспоможение: Родовые сертификаты\n\nh1. Родовые сертификаты\n\n\nh2. Оформление электронного родового сертификата\n\nДля оформления Родового сертификата пользователю должна быть назначена роль «Родовой сертификат».\n{{41158fa0-97a2-4bed-8d06-024e084d5544.png}}\n\nДля формирования талона родового сертификата необходимо внести сведения по договору на оплату услуг заключенного с СФР.  На главной странице Системы выбрать раздел «Родовые сертификаты», затем пункт «Договоры» на панели навигации ЭРС.\n{{85f426ac-4b3e-4942-96ff-0abba83ac427.png}}\n\nДля создания договора следует нажать кнопку "Добавить".На форме создания нового договора следует заполнить обязательное поле «№ договора», заполняется вручную с клавиатуры. Поля «Дата договора», «Срок действия с по» заполняются автоматически текущей датой, поля доступны для редактирования. Поля заполняются путем выбора даты из календаря или вручную с клавиатуры. В блоке «Услуги по договору» необходимо установить флажок в значениях, по кото

In [27]:
replace_abbreviations(docs.loc[35, 'text'], terms)

'WEB Родовспоможение: Родовые сертификаты\n\nh1. Родовые сертификаты\n\n\nh2. Оформление электронного родового сертификата\n\nДля оформления Родового сертификата пользователю должна быть назначена роль «Родовой сертификат».\n{{41158fa0-97a2-4bed-8d06-024e084d5544.png}}\n\nДля формирования талона родового сертификата необходимо внести сведения по договору на оплату услуг заключенного с СФР.  На главной странице Системы выбрать раздел «Родовые сертификаты», затем пункт «Договоры» на панели навигации ЭРС.\n{{85f426ac-4b3e-4942-96ff-0abba83ac427.png}}\n\nДля создания договора следует нажать кнопку "Добавить".На форме создания нового договора следует заполнить обязательное поле «№ договора», заполняется вручную с клавиатуры. Поля «Дата договора», «Срок действия с по» заполняются автоматически текущей датой, поля доступны для редактирования. Поля заполняются путем выбора даты из календаря или вручную с клавиатуры. В блоке «Услуги по договору» необходимо установить флажок в значениях, по кото